# E05：文本文件读取、清洗与结构化

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：综合性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 with open 和 readlines 读取课程提供的 UTF-8 文本文件。
2. 使用 strip、split 和 replace 处理受控的 ASCII 文本问题。
3. 复用函数、循环、判断和异常处理，把文本行转换为结构化记录。
4. 检查记录数、字段、重复值和缺失值，并把结果保存为 JSON。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不填写真实调查信息，不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查结果是否与任务要求一致。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行课程提供的数据设置单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- 如果变量状态混乱，重启内核并从上到下重新运行。
- E02–E04 的同源样本由课程准备代码载入；直接运行即可，不要求解释其中的文件读取代码。

## 建议节奏

- 路径确认、读取和观察：10 分钟
- 单行拆分和字段函数：15 分钟
- 记录解析和批量去重：20 分钟
- 类别、缺失值和记录数核验：15 分钟
- JSON 保存和官方检查点比较：15 分钟
- 调试、实验小结与提交：15 分钟

## 本章第一次读取外部文件

课程数据统一使用 UTF-8；代码中固定写 `encoding="utf-8"`，本实验不展开编码转换。无论从 Notebook 目录还是仓库根目录启动，输出都固定写入 `lab/applied-stat-64/chapter2/e05_output`，不得覆盖 `lab/shared`。

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

chapter2_dir = Path(".").resolve()
if not (chapter2_dir / "e05.ipynb").exists() and not (chapter2_dir / "e05_inclass.ipynb").exists():
    chapter2_dir = Path("lab/applied-stat-64/chapter2").resolve()

shared_dir = chapter2_dir.parents[1] / "shared"

raw_path = shared_dir / "raw" / "student_survey_raw.txt"
official_path = shared_dir / "checkpoints" / "e05_cleaned_records.json"
output_path = chapter2_dir / "e05_output" / "e05_cleaned_records.json"

if not raw_path.exists():
    raw_path = chapter2_dir / "student_survey_raw.txt"
if not official_path.exists():
    official_path = chapter2_dir / "e05_cleaned_records.json"

release_url = "https://rwang0040.github.io/python-for-statistics-public/lab/shared"

# 本地都没有数据文件时，从课程发布网页下载并保存到笔记本同级（需要联网）
try:
    if not raw_path.exists():
        raw_path.write_bytes(urlopen(release_url + "/raw/student_survey_raw.txt", timeout=10).read())
    if not official_path.exists():
        official_path.write_bytes(urlopen(release_url + "/checkpoints/e05_cleaned_records.json", timeout=10).read())
except OSError as error:
    raise RuntimeError(
        "无法从课程发布网页下载数据文件（网络不可用）。请稍后联网重试，或手动把 student_survey_raw.txt 和 e05_cleaned_records.json 下载到本 Notebook 同级目录。"
    ) from error

print("原始文件：", raw_path.resolve())
print("文件是否存在：", raw_path.exists())
print("固定输出位置：", output_path.resolve())

## 任务 1：读取并观察原始文本

使用 `with open(..., encoding="utf-8")` 和 `readlines()` 读取文件，输出总行数、表头和前两条数据行。

In [ ]:
lines = []

# TODO：使用 with open 和 readlines 替换上面的空列表
# with open(...):
#     lines = ...

# TODO：输出总行数、表头和前两条数据行

## 任务 2：手工拆分一条记录

对第一条数据行先 `strip()`，再按逗号 `split()`，检查字段数量及各字段内容。

In [ ]:
# TODO：清理并拆分第一条数据行
# first_line =
# first_parts =
# print(...)

## 任务 3：完成字段清洗函数

课程已经提供缺失标记、类别映射和函数框架。补全关键转换步骤：类别字段统一成数据字典中的规范值；数值字段把 `NA`、`unknown` 和 `-` 转换为 `None`。

In [ ]:
missing_markers = ["", "na", "n/a", "none", "unknown", "-"]


def clean_text(value):
    text = value.strip()
    if text.lower() in missing_markers:
        return None
    return text


def clean_category(value, mapping):
    text = clean_text(value)
    if text is None:
        return None
    key = text.lower()
    if key in mapping:
        return mapping[key]
    return None


def parse_number(value, percent=False):
    text = clean_text(value)
    if text is None:
        return None
    if percent:
        text = text.replace("%", "").strip()
    try:
        # TODO：把清理后的文本转换为浮点数并返回
        return None
    except ValueError:
        return None


gender_mapping = {
    "female": "female",
    "f": "female",
    "male": "male",
    "m": "male",
}
major_mapping = {
    "statistical_computing": "statistical_computing",
    "applied_statistics": "applied_statistics",
    "applied statistics": "applied_statistics",
    "data_science": "data_science",
    "data science": "data_science",
}
prior_mapping = {"yes": "yes", "y": "yes", "no": "no", "n": "no"}
plan_mapping = {
    "regular": "regular",
    "standard": "regular",
    "feedback": "feedback",
    "instant_feedback": "feedback",
}

# TODO：用正常值、类别别名、百分数和缺失标记检查函数

## 任务 4：把一行文本解析成一条记录

课程已经提供字段检查和记录字典框架。补全 8 个字段对应的清洗调用；字段数不正确时返回 `None`。

In [ ]:
def parse_record(line):
    parts = line.strip().split(",")
    if len(parts) != 8:
        return None

    return {
        "student_id": clean_text(parts[0]),
        "alias": clean_text(parts[1]),
        # TODO：参照字段顺序补全其余 6 个字段的清洗调用
    }


# TODO：调用 parse_record() 检查第一条数据行

## 任务 5：批量生成结构化记录并去重

课程已经提供循环与分支骨架。补全计数和列表更新：以清理后的 `student_id` 识别重复记录，保留第一次出现的记录。

In [ ]:
clean_records = []
seen_ids = []
invalid_line_count = 0
duplicate_count = 0

for line in lines[1:]:
    record = parse_record(line)
    if record is None or record["student_id"] is None:
        # TODO：更新无法解析的记录数
        pass
    elif record["student_id"] in seen_ids:
        # TODO：更新重复记录数
        pass
    else:
        # TODO：保存 student_id 和记录
        pass

# TODO：输出清洗后记录数、重复记录数和无法解析的记录数

## 任务 6：检查类别和缺失值

统计学习时长、出勤率的缺失数量，并输出各字段的规范类别集合。

In [ ]:
missing_hours = 0
missing_attendance = 0
genders = []
majors = []
plans = []

for record in clean_records:
    # TODO：统计两个数值字段的缺失数量
    # TODO：分别收集三个类别字段的不重复规范值
    pass

# TODO：输出缺失数量和排序后的类别列表

## 任务 7：保存学生输出

创建 `e05_output` 目录，把 `clean_records` 写为 JSON，再重新读取并检查记录数。

In [ ]:
output_path.parent.mkdir(exist_ok=True)

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(clean_records, file, indent=2)

with open(output_path, "r", encoding="utf-8") as file:
    saved_records = json.load(file)

print("输出文件：", output_path.resolve())
print("重新读取的记录数：", len(saved_records))

## 任务 8：与官方检查点核验

读取只读官方检查点，比较记录数、字段和全部记录。官方文件只用于核验，不得覆盖。

In [ ]:
# TODO：读取 official_path 并比较记录数、字段和全部记录
# 注意：只读取，不覆盖官方检查点

## 实验小结

请用 Markdown 回答：

1. 文件读取成功为什么不等于数据已经可以分析？
2. E02–E04 中的哪些知识在本实验中被重新使用？
3. 为什么学生输出和官方检查点必须分开保存？

### 实验小结

- 